# LangGraph RPG 游戏引擎结构图

这个笔记本生成LangGraph RPG游戏引擎的结构图和说明。

## 项目概述

这是一个基于LangGraph构建的RPG游戏引擎，使用状态机管理游戏流程，支持多角色互动、事件裁判、记忆记录等功能。

In [1]:
import json
import graphviz
from IPython.display import Image, display, Markdown
import os
import sys

# 添加项目路径到系统路径
sys.path.insert(0, os.path.abspath('.'))
sys.path.insert(0, os.path.abspath('./src'))

In [2]:
# 读取配置文件
config_path = 'src/agent/config.py'
with open(config_path, 'r', encoding='utf-8') as f:
    config_content = f.read()

# 解析角色配置
import re

# 查找角色定义
char_match = re.search(r'CHARACTER_CARDS:\s*dict\[str,\s*str\]\s*=\s*({[^}]+})', config_content, re.DOTALL)
if char_match:
    char_dict_str = char_match.group(1)
    # 简单解析角色ID
    char_ids = re.findall(r'"([^"]+)"\s*:', char_dict_str)
    print(f"找到 {len(char_ids)} 个角色: {char_ids}")
else:
    print("未找到角色配置")
    char_ids = []

找到 2 个角色: ['char_001', 'char_002']


In [3]:
# 创建LangGraph结构图
dot = graphviz.Digraph('LangGraph_RPG_Engine', format='png')
dot.attr(rankdir='TB', size='20,15', ratio='auto')
dot.attr('node', shape='box', style='filled', fillcolor='lightblue', fontname='Arial')
dot.attr('edge', fontname='Arial', fontsize='10')

# 定义颜色
colors = {
    'start': 'lightgreen',
    'end': 'lightcoral',
    'main': 'lightblue',
    'parallel': 'lightyellow',
    'conditional': 'lightgray',
    'character': 'lightpink',
}

# 主流程节点
nodes = [
    ('START', '开始', colors['start']),
    ('init_state', '初始化状态', colors['main']),
    ('is_command', '命令判断', colors['conditional']),
    ('handle_command', '命令处理', colors['main']),
    ('dispatcher', '调度器', colors['main']),
    ('event_judge', '事件裁判', colors['main']),
    ('memory_recorder', '记忆记录器', colors['main']),
    ('variable_updater', '变量更新器', colors['main']),
    ('enemy_action', '敌人行动', colors['main']),
    ('should_auto_interact', '自动交互判断', colors['conditional']),
    ('auto_interact_setup', '自动交互设置', colors['main']),
    ('after_auto_variable_updater', '自动后变量更新判断', colors['conditional']),
    ('response_composer', '响应合成器', colors['main']),
    ('END', '结束', colors['end']),
]

# 添加角色节点
for char_id in char_ids:
    node_name = f'char_respond_{char_id}'
    nodes.append((node_name, f'角色响应\n{char_id}', colors['character']))

# 添加所有节点
for node_id, label, color in nodes:
    dot.node(node_id, label, fillcolor=color)

# 主流程边
edges = [
    ('START', 'init_state'),
    ('init_state', 'is_command'),
    ('is_command', 'handle_command', 'command'),
    ('is_command', 'dispatcher', 'game'),
    ('handle_command', 'END'),
    ('dispatcher', 'char_respond_*', '并行所有角色'),
    ('char_respond_*', 'event_judge'),
    ('char_respond_*', 'memory_recorder'),
    ('event_judge', 'variable_updater'),
    ('memory_recorder', 'variable_updater'),
    ('variable_updater', 'after_auto_variable_updater'),
    ('after_auto_variable_updater', 'auto_interact_setup', 'auto_loop'),
    ('after_auto_variable_updater', 'enemy_action', 'proceed'),
    ('enemy_action', 'should_auto_interact'),
    ('should_auto_interact', 'auto_interact_setup', 'continue'),
    ('should_auto_interact', 'response_composer', 'done'),
    ('auto_interact_setup', 'dispatcher'),
    ('response_composer', 'END'),
]

# 添加边
for edge in edges:
    if len(edge) == 2:
        dot.edge(edge[0], edge[1])
    elif len(edge) == 3:
        dot.edge(edge[0], edge[1], label=edge[2])

# 添加角色并行边
for char_id in char_ids:
    node_name = f'char_respond_{char_id}'
    dot.edge('dispatcher', node_name)
    dot.edge(node_name, 'event_judge')
    dot.edge(node_name, 'memory_recorder')

# 渲染并显示
output_path = 'langgraph_structure'
dot.render(output_path, cleanup=True, format='png')

# 显示图像
display(Image(filename=f'{output_path}.png'))

print(f"图形已保存到: {output_path}.png")

ExecutableNotFound: failed to execute WindowsPath('dot'), make sure the Graphviz executables are on your systems' PATH

## LangGraph 工作流程说明

### 1. 初始化流程
1. **START** → **init_state**: 初始化游戏状态
2. **init_state** → **is_command**: 判断用户输入是否为命令

### 2. 命令分支
- 如果是命令: **is_command** → **handle_command** → **END** (直接处理命令并结束)
- 如果是游戏对话: **is_command** → **dispatcher** (进入游戏主流程)

### 3. 游戏主流程
1. **dispatcher**: 调度器决定哪些角色参与本轮对话
2. **并行执行**: dispatcher 同时调用所有参与的角色节点 (char_respond_*)
3. **并行处理**: 所有角色响应后，并行执行:
   - **event_judge**: 事件裁判，判断变量变化和特殊事件
   - **memory_recorder**: 记忆记录器，更新游戏记忆摘要
4. **variable_updater**: 更新游戏变量

### 4. 自动交互循环判断
1. **variable_updater** → **after_auto_variable_updater**: 判断是否还在自动交互循环中
   - 如果是: **auto_loop** → **auto_interact_setup** → **dispatcher** (重新开始循环)
   - 如果否: **proceed** → **enemy_action** (继续主流程)

### 5. 敌人行动和自动交互
1. **enemy_action**: 敌人策略决策
2. **should_auto_interact**: 判断用户是否昏迷需要自动交互
   - 如果昏迷: **continue** → **auto_interact_setup** → **dispatcher** (开始自动交互)
   - 如果正常: **done** → **response_composer** (合成最终响应)

### 6. 响应合成
1. **response_composer**: 合成最终叙事文本
2. **response_composer** → **END**: 结束本轮

In [4]:
# 显示角色配置
print("## 角色配置")
print(f"角色数量: {len(char_ids)}")
for char_id in char_ids:
    print(f"- {char_id}")

# 查找模型配置
model_match = re.search(r'CHARACTER_MODELS:\s*dict\[str,\s*dict\]\s*=\s*({[^}]+})', config_content, re.DOTALL)
if model_match:
    print("\n## 角色模型配置")
    print("每个角色可以配置不同的LLM模型和参数")
    
# 查找组件模型配置
component_match = re.search(r'COMPONENT_MODELS:\s*dict\[str,\s*dict\]\s*=\s*({[^}]+})', config_content, re.DOTALL)
if component_match:
    print("\n## 组件模型配置")
    print("各功能组件使用不同的LLM模型: dispatcher, event_judge, memory_recorder等")

# 查找游戏参数
param_patterns = {
    'MAX_AUTO_INTERACT_TURNS': '昏迷时最大自动交互轮次',
    'ENEMY_ACTION_LUCK_THRESHOLD': '触发敌人行动的最低气运值',
    'ENEMY_ACTION_TURN_COOLDOWN': '敌人行动冷却轮次',
}

print("\n## 游戏参数")
for param, desc in param_patterns.items():
    match = re.search(f'{param}\s*=\s*(\d+)', config_content)
    if match:
        print(f"- {desc}: {match.group(1)}")

## 角色配置
角色数量: 2
- char_001
- char_002

## 角色模型配置
每个角色可以配置不同的LLM模型和参数

## 组件模型配置
各功能组件使用不同的LLM模型: dispatcher, event_judge, memory_recorder等

## 游戏参数
- 昏迷时最大自动交互轮次: 10
- 触发敌人行动的最低气运值: 500
- 敌人行动冷却轮次: 5


<>:28: SyntaxWarning: invalid escape sequence '\s'
<>:28: SyntaxWarning: invalid escape sequence '\s'
C:\Users\yich\AppData\Local\Temp\ipykernel_35732\2687736246.py:28: SyntaxWarning: invalid escape sequence '\s'
  match = re.search(f'{param}\s*=\s*(\d+)', config_content)


## 状态结构 (GameState)

游戏状态包含以下关键组件:

### 核心状态
1. **user_input**: 用户输入
2. **story_time**: 游戏时间线
3. **memory_summary**: 游戏记忆摘要
4. **scene**: 当前场景描述
5. **user_status**: 用户状态 (normal/unconscious/injured)

### 角色状态
1. **character_vars**: 每个角色的变量集合
   - affection: 好感度 (0-100)
   - sensitivity_base: 基础敏感度 (0-100)
   - sensitivity_extra: 对其他角色的额外敏感度
   - luck: 气运值 (0-1000)
   - inventory: 物品库存
   - is_hostile: 是否为敌对角色
   - last_enemy_action_turn: 上次敌人行动轮次

### 流程状态
1. **active_characters**: 本轮活跃角色列表
2. **dispatcher_context**: 调度器为每个角色提供的上下文
3. **auto_interact_turn**: 自动交互计数
4. **pending_effects**: 待处理的特殊效果

## 关键功能组件

### 1. 调度器 (dispatcher)
- 决定哪些角色参与本轮对话
- 为每个角色提供过滤后的上下文信息
- 保持角色信息隔离

### 2. 事件裁判 (event_judge)
- 判断角色变量变化
- 触发特殊事件
- 更新用户状态

### 3. 记忆记录器 (memory_recorder)
- 记录关键情节发展
- 压缩游戏记忆摘要
- 保持摘要简洁(不超过500字)

### 4. 敌人策略 (enemy_strategy)
- 敌对角色的策略决策
- 消耗气运值获取特殊能力
- 触发影响剧情的事件

### 5. 响应合成器 (response_composer)
- 合成最终叙事文本
- 以第二人称("你")叙述
- 整合所有角色行为和事件

### 6. 自动叙事器 (auto_interact)
- 用户昏迷时的自动交互
- 角色间自动对话/行动描述
- 第三人称叙述

In [ ]:
# 创建简化的Mermaid流程图
mermaid_diagram = """```mermaid
graph TB
    %% 颜色定义
    classDef start fill:#90EE90
    classDef end fill:#F08080
    classDef main fill:#ADD8E6
    classDef conditional fill:#D3D3D3
    classDef character fill:#FFB6C1
    
    %% 节点定义
    START[开始]:::start
    init_state[初始化状态]:::main
    is_command{命令判断?}:::conditional
    handle_command[命令处理]:::main
    dispatcher[调度器]:::main
    END[结束]:::end
    
    %% 角色节点 (示例)
    char1[角色响应 char_001]:::character
    char2[角色响应 char_002]:::character
    
    %% 流程节点
    event_judge[事件裁判]:::main
    memory_recorder[记忆记录器]:::main
    variable_updater[变量更新器]:::main
    after_auto{自动循环?}:::conditional
    enemy_action[敌人行动]:::main
    should_auto{用户昏迷?}:::conditional
    auto_interact[自动交互设置]:::main
    response_composer[响应合成器]:::main
    
    %% 边定义
    START --> init_state
    init_state --> is_command
    
    %% 命令分支
    is_command -- 命令 --> handle_command
    handle_command --> END
    
    %% 游戏分支
    is_command -- 游戏 --> dispatcher
    
    %% 角色并行
    dispatcher --> char1
    dispatcher --> char2
    
    %% 事件处理并行
    char1 --> event_judge
    char1 --> memory_recorder
    char2 --> event_judge
    char2 --> memory_recorder
    
    %% 变量更新
    event_judge --> variable_updater
    memory_recorder --> variable_updater
    
    %% 自动循环判断
    variable_updater --> after_auto
    after_auto -- auto_loop --> auto_interact
    after_auto -- proceed --> enemy_action
    
    %% 自动交互设置回到调度器
    auto_interact --> dispatcher
    
    %% 敌人行动和自动交互判断
    enemy_action --> should_auto
    should_auto -- continue --> auto_interact
    should_auto -- done --> response_composer
    
    %% 响应合成
    response_composer --> END
```"""

display(Markdown(mermaid_diagram))

## 配置说明

### 1. 角色卡配置
在 `src/agent/config.py` 中配置:
- **CHARACTER_CARDS**: 角色卡定义，包含角色名称、性别、性格、背景等
- **CHARACTER_INIT_VARS**: 角色初始变量

### 2. 模型配置
- **CHARACTER_MODELS**: 每个角色使用的LLM模型
- **COMPONENT_MODELS**: 各功能组件使用的模型
- 支持多个LLM提供商: OpenAI、Anthropic、Ollama
- 支持第三方中转服务 (如 one-api, openrouter)

### 3. 提示词模板
每个组件都有对应的系统提示词:
- **DISPATCHER_SYSTEM**: 调度器提示词
- **EVENT_JUDGE_SYSTEM**: 事件裁判提示词
- **MEMORY_RECORDER_SYSTEM**: 记忆记录器提示词
- **ENEMY_STRATEGY_SYSTEM**: 敌人策略提示词
- **RESPONSE_COMPOSER_SYSTEM**: 响应合成器提示词
- **AUTO_INTERACT_SYSTEM**: 自动交互提示词

### 4. 游戏参数
- **MAX_AUTO_INTERACT_TURNS**: 昏迷时最大自动交互轮次 (默认: 10)
- **ENEMY_ACTION_LUCK_THRESHOLD**: 触发敌人行动的最低气运值 (默认: 500)
- **ENEMY_ACTION_TURN_COOLDOWN**: 敌人行动冷却轮次 (默认: 5)

## 使用方法

### 1. 安装依赖
```bash
pip install -e . "langgraph-cli[inmem]"
```

### 2. 配置环境变量
复制 `.env.example` 到 `.env` 并设置:
- `OPENAI_BASE_URL`: LLM中转服务地址
- `OPENAI_API_KEY`: API密钥
- `LANGSMITH_API_KEY`: (可选) LangSmith追踪

### 3. 启动服务器
```bash
langgraph dev
```

### 4. 访问LangGraph Studio
浏览器打开: `http://localhost:2024`

### 5. 自定义配置
1. 编辑 `src/agent/config.py` 配置角色和模型
2. 编辑 `src/agent/graph.py` 修改图结构
3. 编辑 `src/agent/nodes.py` 修改节点逻辑

## 注意事项

1. **角色信息隔离**: 调度器确保角色只能看到应该知道的信息
2. **变量范围**: 好感度(0-100)、气运值(0-1000)等都有范围限制
3. **自动交互**: 用户昏迷时最多进行10轮自动交互
4. **敌人行动冷却**: 敌人行动有5轮的冷却时间
5. **记忆摘要**: 记忆记录器保持摘要不超过500字

## 扩展建议

1. **添加新角色**: 在 `CHARACTER_CARDS` 和 `CHARACTER_INIT_VARS` 中添加
2. **自定义事件**: 在 `EVENT_JUDGE_SYSTEM` 中添加新事件类型
3. **添加物品系统**: 扩展 `inventory` 变量支持物品效果
4. **多场景支持**: 扩展 `scene` 变量支持场景切换
5. **添加技能系统**: 为角色添加技能和战斗机制


In [ ]:
# 生成项目结构树
import os

def print_tree(startpath, prefix=""):
    """打印项目结构树"""
    for root, dirs, files in os.walk(startpath):
        # 过滤掉不需要的目录
        dirs[:] = [d for d in dirs if d not in ['.git', '__pycache__', '.venv', 'venv', '.idea', '.vscode']]
        
        level = root.replace(startpath, '').count(os.sep)
        indent = ' ' * 2 * level
        print(f"{indent}{os.path.basename(root)}/")
        subindent = ' ' * 2 * (level + 1)
        
        for f in files:
            if not f.endswith(('.pyc', '.pyo', '.pyd')):
                print(f"{subindent}{f}")

print("## 项目文件结构")
print_tree('.', prefix="")

## 总结

这个LangGraph RPG游戏引擎展示了如何用状态机管理复杂的游戏逻辑:

### 核心优势
1. **模块化设计**: 每个功能组件独立，易于维护和扩展
2. **并行处理**: 角色响应、事件裁判、记忆记录并行执行
3. **状态管理**: 统一的状态管理确保数据一致性
4. **可配置性**: 角色、模型、参数都可灵活配置
5. **可视化调试**: 支持LangGraph Studio可视化调试

### 应用场景
1. **交互式叙事游戏**: 多角色互动的故事驱动游戏
2. **角色扮演模拟**: 角色关系发展和情节推进
3. **教育游戏**: 基于对话的学习和互动体验
4. **创意写作工具**: 辅助故事创作和角色开发

### 技术栈
- **LangGraph**: 状态机和流程管理
- **LangChain**: LLM调用和提示词管理
- **多种LLM支持**: OpenAI、Anthropic、Ollama等
- **JSON格式**: 结构化数据交换
- **环境变量配置**: 灵活的部署配置

---

*注: 此项目为Gemini驱动的初版实现，功能完整但未经充分测试，建议在开发环境中使用和扩展。*